In [1]:
from pyspark.sql import SparkSession
import json, time
import pandas as pd

spark = (SparkSession.builder
    .appName("FastFood-Partie2-RDD")
    .master("spark://spark-master:7077").config("spark.cores.max", "2")
    .config("spark.driver.host", "jupyter")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate())
sc = spark.sparkContext

lines = sc.textFile("s3a://lake/bronze/history/orders_history.jsonl")
orders = lines.map(json.loads)
print("Nombre d'événements :", orders.count())
print(orders.first())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 23:02:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/13 23:02:06 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
[Stage 0:>                                                          (0 + 2) / 2]

Nombre d'événements : 50000
{'order_id': 'ORD-0007147', 'restaurant_id': 'R-179', 'customer_id': 'C-14693', 'city': 'Tanger', 'amount_mad': 69.8, 'items_count': 2, 'distance_km': 3.3, 'courier_id': 'L-TNG-05', 'status': 'CREATED', 'event_time': '2026-11-06T21:47:51Z'}


In [2]:
delivered_raw = orders.filter(lambda e: e["status"] == "DELIVERED")
delivered = (delivered_raw
             .map(lambda e: (e["order_id"], e))
             .reduceByKey(lambda a, b: a)
             .values()
             .cache())
print("DELIVERED avant dédoublonnage :", delivered_raw.count())
print("DELIVERED après dédoublonnage :", delivered.count())

DELIVERED avant dédoublonnage : 15760
DELIVERED après dédoublonnage : 15435


In [3]:
t0 = time.time()
ca_city = (delivered.map(lambda e: (e["city"], e["amount_mad"]))
                    .reduceByKey(lambda a, b: a + b)
                    .collect())
t_rdd_ca = time.time() - t0

t0 = time.time()
sum_count = (delivered.map(lambda e: (e["city"], e["amount_mad"]))
    .aggregateByKey((0.0, 0),
                    lambda acc, v: (acc[0] + v, acc[1] + 1),
                    lambda a, b: (a[0] + b[0], a[1] + b[1])))
panier = sum_count.mapValues(lambda s: s[0] / s[1]).collect()
t_rdd_panier = time.time() - t0

df = pd.DataFrame(sorted(ca_city), columns=["ville", "ca_total_mad"])
df["panier_moyen_mad"] = [dict(panier)[v] for v in df["ville"]]
print(f"Temps RDD : CA {t_rdd_ca:.2f} s ; panier moyen {t_rdd_panier:.2f} s")
df.round(2)

Temps RDD : CA 0.35 s ; panier moyen 0.38 s


,ville,ca_total_mad,panier_moyen_mad
0,Casablanca,694042.69,166.40
1,Marrakech,704863.08,169.11
2,Rabat,596142.96,167.22
3,Tanger,583033.49,165.12


In [4]:
top5 = (delivered.map(lambda e: (e["restaurant_id"], 1))
                 .reduceByKey(lambda a, b: a + b)
                 .takeOrdered(5, key=lambda x: -x[1]))
pd.DataFrame(top5, columns=["restaurant_id", "commandes_livrees"])

,restaurant_id,commandes_livrees
0,R-113,103
1,R-035,99
2,R-004,97
3,R-169,95
4,R-041,94


In [5]:
ca_rdd = (delivered.map(lambda e: (e["city"], e["amount_mad"]))
                   .reduceByKey(lambda a, b: a + b))
print(ca_rdd.toDebugString().decode("utf-8"))

(2) PythonRDD[30] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[29] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[28] at partitionBy at <unknown>:0 []
 +-(2) PairwiseRDD[27] at reduceByKey at /tmp/ipykernel_840/2372130001.py:1 []
    |  PythonRDD[26] at reduceByKey at /tmp/ipykernel_840/2372130001.py:1 []
    |  PythonRDD[8] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 820.9 KiB; DiskSize: 0.0 B
    |  MapPartitionsRDD[7] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[6] at partitionBy at <unknown>:0 []
    +-(2) PairwiseRDD[5] at reduceByKey at /tmp/ipykernel_840/1653688494.py:2 []
       |  PythonRDD[4] at reduceByKey at /tmp/ipykernel_840/1653688494.py:2 []
       |  s3a://lake/bronze/history/orders_history.jsonl MapPartitionsRDD[1] at textFile at <unknown>:0 []
       |  s3a://lake/bronze/history/orders_history.jsonl HadoopRDD[0] at textFile at <unknown>:0 []


In [6]:
spark.stop()